# 1-board: opamp-temperature

Measure how the current chain's gain and zero move with board temperature, so a current read on a warm board is known to the right fraction and the winding thermometer's ratio is not reading the amplifier.

## Purpose

The winding thermometer reads resistance as a ratio to a reference taken at the start of a hold, and copper moves about 0.39 % per degree. A gain that drifts while the board warms through the hold reads as winding temperature. The current limit and every stall decision are in amperes through the same gain. This experiment runs the DC laps of current-scale at four board temperatures and fits the gain and the zero at each.

## Procedure

The board on the same controlled heat source as ntc-curve, on the DPS, with the bare load on its terminals and the meter in series with the load, both off the heater. The load dissipates up to 9 W of its own at the top level, so it stays off the heater and away from the board, and the meter stays at room. At each plateau, room, about 45 C, about 60 C and back to room, `capture.py` runs laps at 4.5, 5.5, 7.2 and 7.6 V in both directions and every row carries the board NTC, so each lap carries its own temperature. The ntc-curve result turns that into degrees.

1. The board and the heater as in ntc-curve, the thermocouple off the board. The DMM on DC A in series with the load, or on DC V across the shunt pads for the `rs1` route.
2. `capture.py`. It asks at each plateau and runs the laps.

## Data

- `<plateau>/levels.csv`, `<plateau>/laps/<k>/sweep.csv`: the laps, `ntc_raw` in every row.
- `dmm.csv`: the meter through the run. `marks.csv`: the plateaus. `route.txt`: `ammeter` or `rs1`.

### The meter through the run

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "opamp-temperature" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", results.rel(CAPTURE))

In [ ]:
dmm = b41t.load(CAPTURE / "dmm.csv"); marks = pd.read_csv(CAPTURE / "marks.csv")
route = (CAPTURE / "route.txt").read_text().strip() if (CAPTURE / "route.txt").exists() else "ammeter"
plateaus = [m for m in marks["mark"] if m != "end" and (CAPTURE / m.replace(" ", "-") / "levels.csv").exists()]
print("route", route, "plateaus", plateaus)
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(dmm["t"], dmm["value"], ".", ms=2, color=BLUE); ax.set_xlabel("time (s)"); ax.set_ylabel("meter"); ax.grid(alpha=0.3)
for _, m in marks.iterrows():
    ax.axvline(m["t_host"] - dmm["t_host"].iloc[0], color=MUTED, lw=0.6); ax.text(m["t_host"] - dmm["t_host"].iloc[0], ax.get_ylim()[1], m["mark"], fontsize=8, va="top")
plt.show()

The meter's log with the plateaus marked. Each plateau is four levels of two laps.

## Method

Each plateau's laps give counts over the rest zero against the meter, as in current-scale, and a straight line through them is that plateau's gain and offset. The board temperature of a plateau is the mean of its laps' NTC readings, converted by the ntc-curve result's refitted beta. The gain at each plateau against the first room plateau is the drift, and read through copper's 0.39 % per degree it is the winding temperature error a hold would make. The zero's move with temperature is read the same way in counts.

### Gain per plateau

In [ ]:
nc = results.read("1-board", "ntc-curve", pair=BOARD.key) if (Path("../ntc-curve/results") / f"{BOARD.key}.json").exists() else None
beta = nc["beta_fit"]["value"] if nc else BOARD.ntc_beta
def ntc_c(raw):
    return 1 / (1 / 298.15 + np.log(BOARD.ntc_ohm(raw) / BOARD.ntc_r25_ohm) / beta) - 273.15
rows, tables = [], {}
for p in plateaus:
    t = laps.table(CAPTURE / p.replace(" ", "-"), BOARD, dmm).dropna(subset=["meter"])
    t["counts"] = t["current_raw"] - t["zero"]; t["x"] = t["meter"].abs() * (1e3 if route == "rs1" else 1)
    tables[p] = t
    f = laps.line(t["x"], t["counts"])
    rows.append({"plateau": p, "board_c": ntc_c(t["ntc_raw"]).mean(), "gain": f["slope"], "gain_se": f["slope_se"],
                 "offset": f["intercept"], "offset_se": f["intercept_se"], "zero": t["zero"].mean(), "laps": len(t)})
P = pd.DataFrame(rows)
P["gain_pct"] = (P["gain"] / P["gain"].iloc[0] - 1) * 100
P["copper_c"] = P["gain_pct"] / 0.39
show(P, n=None)
unit = "counts per mV" if route == "rs1" else "counts per A"
print(f"gain in {unit}; gain_pct against the first plateau; copper_c is that drift read as winding temperature")

One row per plateau: the board temperature from the NTC, the gain and offset of the line through its laps, the torque-off zero, and the gain's drift against the first plateau as a percentage and as the copper temperature it would read as.

In [ ]:
ok = P.dropna(subset=["board_c"])
g = laps.line(ok["board_c"], ok["gain_pct"]) if len(ok) >= 3 else None
z = laps.line(ok["board_c"], ok["zero"]) if len(ok) >= 3 else None
if g:
    print(f"gain {g['slope']:+.4f} % per C ({g['slope_se']:.4f}), zero {z['slope']:+.2f} counts per C over {ok['board_c'].min():.0f} to {ok['board_c'].max():.0f} C")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].errorbar(P["board_c"], P["gain_pct"], yerr=P["gain_se"] / P["gain"] * 100, fmt="o", color=BLUE); ax[0].axhspan(-0.39 * 2, 0.39 * 2, color=MUTED, alpha=0.15)
ax[0].set_xlabel("board (C)"); ax[0].set_ylabel("gain against the first plateau (%)"); ax[0].set_title("the band is 2 C of copper"); ax[0].grid(alpha=0.3)
for p, t in tables.items():
    ax[1].plot(t["x"], t["counts"], "o", ms=4, label=p)
ax[1].set_xlabel("meter (mV)" if route == "rs1" else "meter (A)"); ax[1].set_ylabel("counts over the rest zero"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
ax[2].plot(P["board_c"], P["zero"], "o", color=BLUE); ax[2].set_xlabel("board (C)"); ax[2].set_ylabel("torque-off zero (counts)"); ax[2].grid(alpha=0.3)
plt.show()

Left, the gain at each plateau against the first, with a band of two degrees of copper for scale. Middle, every lap of every plateau on one line. Right, the torque-off zero against board temperature.

## Results

The gain's drift per degree with its standard error as the bounds, the zero's drift per degree, and the range covered.

In [ ]:
results.write("1-board", "opamp-temperature", CAPTURE, pair=BOARD.key,
              notes=f"DC laps at four board temperatures on a controlled heat source, meter route {route}; gain per plateau as a line through counts against the meter",
              gain_drift_pct_per_c=results.quantity(g["slope"], lo=g["slope"] - g["slope_se"], hi=g["slope"] + g["slope_se"], unit="%/C") if g else results.quantity(np.nan, unit="%/C"),
              zero_drift_counts_per_c=results.quantity(z["slope"], lo=z["slope"] - z["slope_se"], hi=z["slope"] + z["slope_se"], unit="counts/C") if z else results.quantity(np.nan, unit="counts/C"),
              gain_drift_max_pct=results.quantity(P["gain_pct"].abs().max(), unit="%"),
              range_c=results.quantity(P["board_c"].max(), lo=P["board_c"].min(), hi=P["board_c"].max(), unit="C"))

## Limitations

- On the ammeter route the meter is off the heater and reads the whole loop's current, so the drift is the chain's as a whole: the shunt's own temperature coefficient is in it. The `rs1` route takes the shunt out.
- The load warms through each plateau's laps; the meter carries that, the ratio does not care.
- Four plateaus over room to 60 C. A slope from four points has the standard error it has.

## Open questions

- Whether the drift is the amplifier's or the shunt's. The two routes back to back at one hot plateau separate them.

## Conclusion

To be written from the capture.